# 🧠 고전 CNN 모델 실습: LeNet-5 / VGG16

이번 노트북에서는 고전적인 CNN 모델 2종을 실습합니다.

1. **LeNet-5** — MNIST 직접 학습 및 평가
2. **VGG16** — ImageNet 사전학습 가중치로 예측 실습

모든 모델의 파라미터 수를 비교하고, 구조를 살펴봅니다.

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, datasets, utils
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input, decode_predictions
import numpy as np
import matplotlib.pyplot as plt
print('TensorFlow version:', tf.__version__)

TensorFlow version: 2.21.0


## [1] LeNet-5 스타일 CNN 구현 및 MNIST 학습

In [2]:
(x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)
print('Train:', x_train.shape, ' Test:', x_test.shape)

Train: (60000, 28, 28, 1)  Test: (10000, 28, 28, 1)


In [3]:
def build_lenet5():
    model = models.Sequential([
        layers.Conv2D(6, (5,5), activation='tanh', input_shape=(28,28,1), padding='same'),
        layers.AveragePooling2D(pool_size=(2,2)),
        layers.Conv2D(16, (5,5), activation='tanh'),
        layers.AveragePooling2D(pool_size=(2,2)),
        layers.Flatten(),
        layers.Dense(120, activation='tanh'),
        layers.Dense(84, activation='tanh'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

lenet5 = build_lenet5()
lenet5.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 14, 14, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 5, 5, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 120)            │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
history = lenet5.fit(x_train, y_train, epochs=5, batch_size=128, validation_split=0.1, verbose=2)

Epoch 1/5
422/422 - 9s - 22ms/step - accuracy: 0.8960 - loss: 0.3630 - val_accuracy: 0.9662 - val_loss: 0.1313
Epoch 2/5
422/422 - 8s - 18ms/step - accuracy: 0.9612 - loss: 0.1296 - val_accuracy: 0.9728 - val_loss: 0.0899
Epoch 3/5
422/422 - 8s - 18ms/step - accuracy: 0.9741 - loss: 0.0865 - val_accuracy: 0.9755 - val_loss: 0.0792
Epoch 4/5
422/422 - 10s - 23ms/step - accuracy: 0.9804 - loss: 0.0643 - val_accuracy: 0.9823 - val_loss: 0.0626
Epoch 5/5
422/422 - 10s - 23ms/step - accuracy: 0.9847 - loss: 0.0495 - val_accuracy: 0.9820 - val_loss: 0.0594


In [5]:
test_loss, test_acc = lenet5.evaluate(x_test, y_test, verbose=0)
print(f"LeNet-5 테스트 정확도: {test_acc:.4f}")

LeNet-5 테스트 정확도: 0.9792


## [2] VGG16 (ImageNet 사전학습 가중치)

In [6]:
vgg16 = VGG16(weights='imagenet')
vgg16.summary()

Model: "vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 4096)           │   102,764,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 1000)           │     4,097,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 138,357,544 (527.79 MB)

 Trainable params: 138,357,544 (527.79 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
img_path = tf.keras.utils.get_file('elephant.jpg', 'https://cdn.hackaday.io/images/1219231493836853483.jpg')
img_vgg = image.load_img(img_path, target_size=(224, 224))
x_vgg = image.img_to_array(img_vgg)
x_vgg = np.expand_dims(x_vgg, axis=0)
x_vgg = preprocess_input(x_vgg)

preds_vgg = vgg16.predict(x_vgg)
print('VGG16 예측 결과:', decode_predictions(preds_vgg, top=3)[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 401ms/step
VGG16 예측 결과: [('n02504458', 'African_elephant', np.float32(0.8433748)), ('n01871265', 'tusker', np.float32(0.15428466)), ('n02504013', 'Indian_elephant', np.float32(0.0023405135))]


## [3] 모델별 파라미터 수 비교

In [8]:
models_info = {
    'LeNet-5 (MNIST)': lenet5.count_params(),
    'VGG16 (ImageNet)': vgg16.count_params()
}
for name, params in models_info.items():
    print(f'{name}: {params:,} parameters')

LeNet-5 (MNIST): 61,706 parameters
VGG16 (ImageNet): 138,357,544 parameters


---
## 📝 [실습 7-2 수행 결과 정리]

> 실행 환경: TensorFlow 2.21.0 / Keras 3.15.1 (CPU 2코어). 코드는 원본 그대로 실행했습니다. 노트북 출력에 없는 값(층별 연산량, 혼동 쌍, 추론 시간)은 학습·로드가 끝난 모델로 따로 측정했습니다.

### 1) LeNet-5 — MNIST 학습 (tanh + AveragePooling, 5 Epoch)

| Epoch | Train Acc | Val Acc | Val Loss |
|:---:|:---:|:---:|:---:|
| 1 | 0.8960 | 0.9662 | 0.1313 |
| 2 | 0.9612 | 0.9728 | 0.0899 |
| 3 | 0.9741 | 0.9755 | 0.0792 |
| 4 | 0.9804 | 0.9823 | 0.0626 |
| 5 | 0.9847 | 0.9820 | 0.0594 |

- **테스트 정확도 0.9792** (교수님 원본 0.9813, 차이는 초기값 차이). 오분류 208장 / 10,000장.
- 가장 많이 헷갈린 쌍은 5→3(19장), 4→9(13장), 2→7(11장), 9→4(9장)입니다. 클래스별 재현율도 숫자 5(95.3%)와 2(96.9%)가 가장 낮습니다. 획 일부만 다른 숫자끼리 혼동하므로, 제조 검사로 치면 **형태가 비슷한 결함 유형끼리의 오분류**에 해당합니다.
- 실습 7-1의 CNN(121,930개, 3 Epoch, 0.9853)과 비교하면 LeNet-5는 **파라미터 절반(61,706개)**으로 0.979에 도달합니다. 필터 수가 6·16개로 적고(7-1은 32·64개) 활성화가 tanh라 같은 Epoch에서의 정확도는 조금 낮습니다(Epoch 1 Val 0.966 vs 0.979).

### 2) VGG16 — ImageNet 사전학습 가중치로 예측

| 순위 | 클래스 | 확률 |
|:---:|:---|:---:|
| 1 | African_elephant | **0.8434** |
| 2 | tusker | 0.1543 |
| 3 | Indian_elephant | 0.0023 |

- 교수님 원본 출력과 소수점 넷째 자리까지 같습니다. 같은 가중치·같은 이미지로 추론만 하므로 결과가 결정적입니다.
- 입력은 200×231 크기의 아프리카코끼리 사진이고, `load_img(target_size=(224, 224))`가 비율을 무시하고 224×224로 늘려서 넣습니다. 2위 tusker(엄니가 큰 코끼리)까지 상위 3개가 모두 코끼리 계열입니다.
- 참고: 이 셀이 받은 `elephant.jpg`는 `~/.keras/datasets/`에 캐시되고, **실습 7-3이 같은 파일명으로 이 캐시를 재사용**합니다(7-3 정리 참고).

### 3) 파라미터 vs 연산량 — "파라미터는 FC, 연산은 Conv"

| 모델 | 구분 | 파라미터 | 비중 | 연산량 (MAC, 1장) | 비중 |
|:---|:---|---:|:---:|---:|:---:|
| LeNet-5 | Conv 2층 | 2,572 | 4.2% | 357,600 | **85.9%** |
|  | FC 3층 | 59,134 | **95.8%** | 58,920 | 14.1% |
| VGG16 | Conv 13층 | 14,714,688 | 10.6% | 약 153.5억 (15.35 G) | **99.2%** |
|  | FC 3층 | 123,642,856 | **89.4%** | 약 1.24억 (0.12 G) | 0.8% |

- VGG16의 첫 FC 층(fc1, 7×7×512 → 4096) 하나가 **1억 276만 개(전체의 74%)**입니다. 528 MB 모델 크기의 대부분이 FC입니다.
- 반대로 계산 시간은 Conv가 지배합니다. Conv는 가중치가 적어도 **모든 위치에서 반복 계산**(가중치 공유)되기 때문입니다. 예를 들어 block1_conv2는 파라미터 3.7만 개지만 연산량은 18.5억 MAC(전체의 12%)입니다.
- 즉 **모델 크기(메모리)는 FC가, 추론 속도는 Conv가 결정**합니다. 7-3의 InceptionV3·ResNet50은 FC를 Global Average Pooling으로 바꿔 파라미터를 2,400~2,600만 개로 줄였습니다.
- CPU(2코어) 추론 시간: VGG16 1장 약 **0.22초**(초당 약 4.6장, `tf.function` 컴파일 기준), LeNet-5 1장 약 **0.05 ms**(1,000장 배치 49 ms). 실시간 라인 검사에 VGG16급 모델을 쓰려면 GPU나 경량화가 필요합니다.

### 4) 핵심 정리
1. LeNet-5(1998)는 Conv–Pool 교차 후 FC로 분류하는 CNN의 기본 틀을 6만 개 파라미터로 보여 주며, MNIST에서 98%에 도달합니다.
2. VGG16은 3×3 Conv를 13층 깊게 쌓아 표현력을 키웠지만, 파라미터 1.38억 개 중 89%가 FC에 몰려 있습니다.
3. 파라미터 수(메모리)와 연산량(속도)은 서로 다른 층이 결정합니다. 제조 현장의 엣지 장비에 올릴 모델을 고를 때는 두 지표를 따로 확인해야 합니다.